In [7]:
from typing import List, Dict, Union, Any, Tuple
import numpy as np
import tensorflow as tf
import random
import datetime
import time
import json
import csv
from collections import deque
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
import secrets

In [8]:
class Fan:
    def __init__(self, fan_speed: int, location: str, *args, **kwargs):
        self.fan_speed = fan_speed 
        self.location = location
    def control_fan(self, speed: int):
        # Implementation to control fan speed
        self.Fan_Speed = speed
    def fetch_status(self) -> dict:
        # Implementation to fetch current status
        return {"Fan_Speed": self.Fan_Speed, "Location": self.location}

In [52]:
class AIAgent:
    def __init__(self,model_type: str,*args, **kwargs):
        self.model_type = model_type
        self.deep_learning_model = None

        if model_type == 'deep_learning':
            # If it's a Deep Learning model, initialize the model
            self.deep_learning_model = self.build_deep_learning_model()

    
    def build_deep_learning_model(self):
        input_dim = 10
        output_dim =2
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(input_dim,)),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(output_dim, activation='softmax')
        ])
        model.compile(optimizer='adam',loss='categorical_crossentropy')
        
        return model
    
    def train(self,dataset : List[Dict[str,Any]]):
        self.train_deep_learning_model(dataset)
        
        
            

    def train_deep_learning_model(self, train_data, train_labels):
        self.deep_learning_model.compile(optimizer='adam',
                              loss='categorical_crossentropy',
                              metrics=['accuracy'])
        self.deep_learning_model.fit(train_data, train_labels, epochs=10)

In [10]:
class Aggregator:
    def __init__(self, temperature, time, motion, AC_temp_status, *args, **kwargs):
        self.temperature = temperature
        self.time = time
        self.motion = motion
        self.AC_temp_status = AC_temp_status
        self.fan_objects = []
        self.ai_agent = None
        self.ai_output_log = []

    def define_fan_objects(self, fan_info: List[Dict[str, Union[int, str]]]):
        for info in fan_info:
            fan_obj = Fan(Fan_Speed=info["Fan_Speed"], location=info["Location"])
            self.fan_objects.append(fan_obj)

    def define_ai_object(self, ai_params: dict):
        self.ai_agent = AIAgent(model_type=ai_params.get('model_type', 'deep_learning'))
        return self.ai_agent

    def invoke_fan_control(self, fan_obj: Fan):
        # Invoke fan control based on AI predictions
        if self.ai_agent:
            state, confidence = self.ai_agent.predict({"temperature": self.temperature, "motion": self.motion})
            if state == 1 and confidence >= 0.85:
                fan_obj.control_fan(5)  # Example: Set fan speed to 5 if predicted state is ON with confidence >= 85%

    def save_ai_output_to_csv(self, ai_output: str):
        # Save AI output to CSV for the last 15 days
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.ai_output_log.append({"timestamp": timestamp, "output": ai_output})

        # Save the last 15 days' data
        last_15_days_data = [log for log in self.ai_output_log if datetime.now() - datetime.strptime(log["timestamp"], "%Y-%m-%d %H:%M:%S") <= timedelta(days=15)]

        with open('ai_output_log.csv', 'w', newline='') as csvfile:
            fieldnames = ["timestamp", "output"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(last_15_days_data)

    def compute_difference(self, previous_state: dict, current_state: dict) -> bool:
        # Compute the difference between previous and current states
        difference = any(previous_state.get(key) != current_state.get(key) for key in current_state)
        manual_override = difference
        return manual_override

    def compute_future_difference(self, current_state: dict, next_predicted_state: dict) -> bool:
        # Compute the difference between current and predicted states
        difference = any(current_state.get(key) != next_predicted_state.get(key) for key in current_state)
        return difference

    def send_signal(self):
        # Placeholder implementation for sending a signal
        print("Sending signal")
    
    
    def fetch_status(self, fan_objects: List[Fan]):
        statuses = []
        for fan in fan_objects:
            statuses.append(fan.fetch_status())
        return statuses

In [54]:
from typing import List, Dict, Any
import numpy as np

class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type

        if model_type == 'ReinforcementLearning':
            # Initialize Q-table for Q-learning
            self.q_table = {}

    def define_reward_function(self, confidence: float) -> float:
        #  reward function for the RL model (Q-learning)
        # Example: Higher confidence results in a higher reward
        return confidence if confidence is not None else 0.0  # Default reward

    def train_rl(self, dataset: List[Dict[str, Any]]):
        # Train the RL model (Q-learning)
        for sample in dataset:
            state = tuple(sample['input'].values())
            action = sample['label']['fan_state']
            confidence = sample['label']['confidence']
            reward = self.define_reward_function(confidence)

            # Initialize Q-value for the state-action pair if not exists
            if state not in self.q_table:
                self.q_table[state] = np.zeros(2)  # Assuming 2 actions (0 for OFF, 1 for ON)

            # Update Q-value using the Q-learning update rule
            if reward is not None:
                self.q_table[state][action] += reward

    def act(self, input_data: Dict[str, Any]) -> str:
        # Take an action based on the current state using the trained Q-table
        state = tuple(input_data.values())
        
        # Choose action with the highest Q-value
        action = np.argmax(self.q_table.get(state, [0, 0]))

        return 'ON' if action == 1 else 'OFF'

# Function to create a dummy dataset for RL training
def create_dummy_dataset(num_samples: int) -> List[Dict[str, Any]]:
    dummy_dataset = []
    for _ in range(num_samples):
        # Create a dummy sample with input and label
        dummy_sample = {
            'input': {'temperature': np.random.uniform(20, 30), 'time': np.random.randint(0, 24),
                      'motion': np.random.choice([True, False]), 'AC_temp_status': np.random.uniform(20, 25)},
            'label': {'fan_state': np.random.choice([0, 1]), 'confidence': np.random.uniform(0.7, 1.0)}
        }
        dummy_dataset.append(dummy_sample)
    return dummy_dataset

# Example: Create an RL agent and train it on a dummy dataset
rl_agent = AIAgent(model_type='ReinforcementLearning')

# Create a dummy dataset for RL training
dummy_rl_dataset = create_dummy_dataset(100)

# Train the RL model
rl_agent.train_rl(dummy_rl_dataset)

# Example: Act based on the current state
current_state = {'temperature': 25.0, 'time': 15, 'motion': True, 'AC_temp_status': 22.0}
action = rl_agent.act(current_state)
print(f"Recommended action based on RL: {action}")


Recommended action based on RL: OFF


In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from typing import List, Dict, Any

class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type

        if model_type == 'ReinforcementLearning':
            # Initialize PPO model
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(11,)),  # Adjust input shape to match the number of features
            tf.keras.layers.Dense(2, activation='softmax')  # Assuming binary action space (ON or OFF)
        ])
        return model

    def train_rl(self, dataset: List[Dict[str, Any]]):
        # Train the PPO model
        states = np.array([list(sample['input'].values()) for sample in dataset])
        actions = np.array([sample['label']['fan_state'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['label']['confidence']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions, num_classes=2)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def act(self, input_data: Dict[str, Any]) -> str:
        # Take an action based on the current state using the trained policy network
        state = np.array(list(input_data.values())).reshape(1, -1)
        policy_prob = self.policy(state, training=False).numpy()
        action = np.random.choice([0, 1], p=policy_prob.flatten())

        return 'ON' if action == 1 else 'OFF'

    def define_advantage(self, confidence: float) -> float:
        # Define advantage based on confidence 
        return confidence
    
def create_dummy_dataset(num_samples: int) -> List[Dict[str, Any]]:
    dummy_dataset = []
    for _ in range(num_samples):
        # Create a dummy sample with input and label
        dummy_sample = {
            'input': {'temperature': np.random.uniform(20, 30), 'time': np.random.randint(0, 24),
                      'motion': np.random.choice([True, False]), 'AC_temp_status': np.random.uniform(20, 25)},
            'label': {'fan_state': np.random.choice([0, 1]), 'confidence': np.random.uniform(0.7, 1.0)}
        }
        dummy_dataset.append(dummy_sample)
    return dummy_dataset
rl_agent = AIAgent(model_type='ReinforcementLearning')

# Create a dummy dataset for RL training
dummy_rl_dataset = create_dummy_dataset(100)

# Train the RL model
rl_agent.train_rl(dummy_rl_dataset)

ppo_agent = AIAgent(model_type='ReinforcementLearning')

# Train the PPO model
ppo_agent.train_rl(dummy_rl_dataset)

current_state = {'temperature': 25.0, 'time': 15, 'motion': True, 'AC_temp_status': 22.0}
action = ppo_agent.act(current_state)
print(f"Recommended action based on PPO: {action}")


ValueError: Input 0 of layer "sequential" is incompatible with the layer: expected shape=(None, 11), found shape=(100, 4)

In [17]:
from typing import List, Dict, Union, Any, Tuple
import random
import datetime
import time
import json
import csv
from collections import deque
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
import secrets
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

#class Fan
class Fan:
    def __init__(self, fan_speed: int, location: str, *args, **kwargs):
        self.fan_speed = fan_speed 
        self.location = location
    def control_fan(self, speed: int):
        #  to control fan speed
        self.Fan_Speed = speed
    def fetch_status(self) -> dict:
        # to fetch current status
        return {"Fan_Speed": self.Fan_Speed, "Location": self.location}
#Aggregator class

class Aggregator:
    def __init__(self, temperature, time, motion, AC_temp_status, *args, **kwargs):
        self.temperature = temperature
        self.time = time
        self.motion = motion
        self.AC_temp_status = AC_temp_status
        self.fan_objects = []
        self.ai_agent = None
        self.ai_output_log = []

    def define_fan_objects(self, fan_info: List[Dict[str, Union[int, str]]]):
        for info in fan_info:
            fan_obj = Fan(Fan_Speed=info["Fan_Speed"], location=info["Location"])
            self.fan_objects.append(fan_obj)

    def define_ai_object(self, ai_params: dict) -> AIAgent:
        self.ai_agent = AIAgent(model_type=ai_params.get('model_type', 'deep_learning'))
        return self.ai_agent

    def invoke_fan_control(self, fan_obj: Fan):
        # fan control based on AI predictions
        if self.ai_agent:
            state, confidence = self.ai_agent.predict({"temperature": self.temperature, "motion": self.motion})
            if state == 1 and confidence >= 0.85:
                fan_obj.control_fan(5)  # Set fan speed to 5 if predicted state is ON with confidence >= 85%

    def save_ai_output_to_csv(self, ai_output: str):
        # Save output to CSV for the last 15 days
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.ai_output_log.append({"timestamp": timestamp, "output": ai_output})

        # Save the last 15 days' data
        last_15_days_data = [log for log in self.ai_output_log if datetime.now() - datetime.strptime(log["timestamp"], "%Y-%m-%d %H:%M:%S") <= timedelta(days=15)]

        with open('ai_output_log.csv', 'w', newline='') as csvfile:
            fieldnames = ["timestamp", "output"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(last_15_days_data)

    def compute_difference(self, previous_state: dict, current_state: dict) -> bool:
        #  difference between previous and current states
        difference = any(previous_state.get(key) != current_state.get(key) for key in current_state)
        manual_override = difference
        return manual_override

    def compute_future_difference(self, current_state: dict, next_predicted_state: dict) -> bool:
        #  difference between current and predicted states
        difference = any(current_state.get(key) != next_predicted_state.get(key) for key in current_state)
        return difference

    def send_signal(self):
        #  sending a signal
        print("Sending signal")
    
    
    def fetch_status(self, fan_objects: List[Fan]):
        statuses = []
        for fan in fan_objects:
            statuses.append(fan.fetch_status())
        return statuses



class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type

        if model_type == 'ReinforcementLearning':
            # Initialize PPO model
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(4,)),  # Assuming 4 features in the state
            tf.keras.layers.Dense(2, activation='softmax')  # Assuming binary action space (ON or OFF)
        ])
        return model

    def train_rl(self, dataset: List[Dict[str, Any]]):
        # Train the PPO model
        states = np.array([list(sample['input'].values()) for sample in dataset])
        actions = np.array([sample['label']['fan_state'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['label']['confidence']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions, num_classes=2)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def act(self, input_data: Dict[str, Any]) -> str:
        # Take an action based on the current state using the trained policy network
        state = np.array(list(input_data.values())).reshape(1, -1)
        policy_prob = self.policy(state, training=False).numpy()
        action = np.random.choice([0, 1], p=policy_prob.flatten())

        return 'ON' if action == 1 else 'OFF'

    def define_advantage(self, confidence: float) -> float:
       
        return confidence
    
    
ppo_agent = AIAgent(model_type='ReinforcementLearning')

# Train the PPO model
ppo_agent.train_rl(dummy_rl_dataset)

current_state = {'temperature': 25.0, 'time': 15, 'motion': True, 'AC_temp_status': 22.0}
action = ppo_agent.act(current_state)
print(f"Recommended action based on PPO: {action}")   

Recommended action based on PPO: ON


In [39]:
from typing import List, Dict, Union, Any, Tuple
import random
import datetime
import time
import json
import csv
from collections import deque
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
import secrets
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

#class Fan
class Fan:
    def __init__(self, fan_speed: int, location: str, *args, **kwargs):
        self.fan_speed = fan_speed 
        self.location = location
    def control_fan(self, speed: int):
        #  to control fan speed
        self.Fan_Speed = speed
    def fetch_status(self) -> dict:
        # to fetch current status
        return {"Fan_Speed": self.Fan_Speed, "Location": self.location}
#Aggregator class

class Aggregator:
    def __init__(self, temperature, time, motion, AC_temp_status, *args, **kwargs):
        self.temperature = temperature
        self.time = time
        self.motion = motion
        self.AC_temp_status = AC_temp_status
        self.fan_objects = []
        self.ai_agent = None
        self.ai_output_log = []

    def define_fan_objects(self, fan_info: List[Dict[str, Union[int, str]]]):
        for info in fan_info:
            fan_obj = Fan(Fan_Speed=info["Fan_Speed"], location=info["Location"])
            self.fan_objects.append(fan_obj)

    def define_ai_object(self, ai_params: dict) -> AIAgent:
        self.ai_agent = AIAgent(model_type=ai_params.get('model_type', 'deep_learning'))
        return self.ai_agent

    def invoke_fan_control(self, fan_obj: Fan):
        # fan control based on AI predictions
        if self.ai_agent:
            state, confidence = self.ai_agent.predict({"temperature": self.temperature, "motion": self.motion})
            if state == 1 and confidence >= 0.85:
                fan_obj.control_fan(5)  # Set fan speed to 5 if predicted state is ON with confidence >= 85%

    def save_ai_output_to_csv(self, ai_output: str):
        # Save output to CSV for the last 15 days
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.ai_output_log.append({"timestamp": timestamp, "output": ai_output})

        # Save the last 15 days' data
        last_15_days_data = [log for log in self.ai_output_log if datetime.now() - datetime.strptime(log["timestamp"], "%Y-%m-%d %H:%M:%S") <= timedelta(days=15)]

        with open('ai_output_log.csv', 'w', newline='') as csvfile:
            fieldnames = ["timestamp", "output"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(last_15_days_data)

    def compute_difference(self, previous_state: dict, current_state: dict) -> bool:
        #  difference between previous and current states
        difference = any(previous_state.get(key) != current_state.get(key) for key in current_state)
        manual_override = difference
        return manual_override

    def compute_future_difference(self, current_state: dict, next_predicted_state: dict) -> bool:
        #  difference between current and predicted states
        difference = any(current_state.get(key) != next_predicted_state.get(key) for key in current_state)
        return difference

    def send_signal(self):
        #  sending a signal
        print("Sending signal")
    
    
    def fetch_status(self, fan_objects: List[Fan]):
        statuses = []
        for fan in fan_objects:
            statuses.append(fan.fetch_status())
        return statuses



class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type

        if model_type == 'ReinforcementLearning':
            # Initialize PPO model
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(4,)),  # Assuming 4 features in the state
            tf.keras.layers.Dense(2, activation='softmax')  # Assuming binary action space (ON or OFF)
        ])
        return model

    def train_rl(self, dataset: List[Dict[str, Any]]):
        # Train the PPO model
        states = np.array([list(sample['input'].values()) for sample in dataset])
        actions = np.array([sample['label']['fan_state'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['label']['confidence']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions, num_classes=2)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def act(self, input_data: Dict[str, Any]) -> str:
        # Take an action based on the current state using the trained policy network
        state = np.array(list(input_data.values())).reshape(1, -1)
        policy_prob = self.policy(state, training=False).numpy()
        action = np.random.choice([0, 1], p=policy_prob.flatten())

        return 'ON' if action == 1 else 'OFF'

    def define_advantage(self, confidence: float) -> float:
       
        return confidence
    
    
ppo_agent = AIAgent(model_type='ReinforcementLearning')

# Train the PPO model
ppo_agent.train_rl(dummy_rl_dataset)

current_state = {'temperature': 25.0, 'time': 15, 'motion': True, 'AC_temp_status': 22.0}
action = ppo_agent.act(current_state)
print(f"Recommended action based on PPO: {action}")   
    


"""class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type

        if model_type == 'ReinforcementLearning':
            # Initialize PPO model
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(10,)),  # Assuming 11 features in the state
            tf.keras.layers.Dense(2, activation='softmax')  # Assuming binary action space (ON or OFF)
        ])
        return model
  

    def act(self, state: Dict[str, Any]) -> int:
        # Convert state to numpy array
        state_array = np.array(list(state.values()))
        state_array = np.expand_dims(state_array, axis=0)

        # Predict action probabilities
        policy_prob = self.policy(state_array, training=False)

        #  the action with the highest probability
        action = np.argmax(policy_prob[0])

        return 'ON' if action == 1 else 'OFF'

    def train_rl(self, dataset: List[Dict[str, Any]]):
    # Train the PPO model
        states = np.array([list(sample['input'].values()) for sample in dataset])
        print(states.shape)
        actions = np.array([sample['TargetFanState'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['confidence']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions, num_classes=2)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def define_advantage(self, confidence: float) -> float:
        return confidence

dummy_rl_dataset = [
    {
        'input': {
            'Temperature': 25.0,
            'TimeOfDay': 15.0,
            'Date': 5,
            'Month': 1,
            'DayOfWeek': 3,
            'MotionSensorState': 1,
            'ACTemperatureStatus': 0,
            'CurrentFanSpeed': 7,
            'Location': 2,
            'PreviousFanState': 1,
        },
        'TargetFanState': 1,  # Assuming the target fan state is ON
        'confidence': 0.9  
    }
]


ppo_agent = AIAgent(model_type='ReinforcementLearning')
ppo_agent.train_rl(dummy_rl_dataset)


current_state = {
    'Temperature': 25.0,
    'TimeOfDay': 15.0,
    'Date': 5,
    'Month': 1,
    'DayOfWeek': 3,
    'MotionSensorState': 1,
    'ACTemperatureStatus': 0,
    'CurrentFanSpeed': 7,
    'Location': 2,
    'PreviousFanState': 1,
}

action = ppo_agent.act(current_state)
print(f"Recommended action based on PPO: {action}")

        
      
        





import random
from datetime import datetime, timedelta

# Function to generate dummy dataset for Fan class Deep Learning model
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Function to create a dummy dataset
def create_dummy_dataset(num_samples=1000):
    np.random.seed(42)

    # Generate random data for the environment variables
    temperature = np.random.uniform(18.0, 30.0, size=num_samples)
    time_of_day = np.random.uniform(0.0, 24.0, size=num_samples)
    date = np.random.randint(1, 31, size=num_samples) # Assuming 31 days in a month 
    month = np.random.choice(12, size=num_samples)
    day_of_week = np.random.choice([2,3,4,5,6,7,1],#monday -2,tuesday-3,wednesday-4,thursday-5,friday-6,saturday-7,sunday-1
                                   size=num_samples)
    motion_sensor_state = np.random.choice([1,0], size=num_samples)#1-True,0-False
    ac_temperature_status = np.random.choice([1,0], size=num_samples) 

    # Generate random data for the fan variables
    current_fan_speed = np.random.randint(0, 11, size=num_samples)
    location = np.random.choice([1,2,3], size=num_samples) #1-Living Room,2-Bedroom,3-Kitchen
    previous_fan_state = np.random.choice([1,0,-1], size=num_samples)

    # Generate random data for the target variable
    target_fan_state = np.random.choice([1,0], size=num_samples)

    # Create a DataFrame
    df = pd.DataFrame({
        'Temperature': temperature,
        'TimeOfDay': time_of_day,
        'Date': date,
        'Month': month,
        'DayOfWeek': day_of_week,
        'MotionSensorState': motion_sensor_state,
        'ACTemperatureStatus': ac_temperature_status,
        'CurrentFanSpeed': current_fan_speed,
        'Location': location,
        'PreviousFanState': previous_fan_state,
        'TargetFanState': target_fan_state
    })

    return df

# Create a dummy dataset
dummy_dataset = create_dummy_dataset()

# Display the first few rows of the dataset
print(dummy_dataset.head())


# Create a dummy dataset
dummy_dataset = create_dummy_dataset()


df = dummy_dataset
X = df.drop('TargetFanState', axis=1)
y = df['TargetFanState']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Build a deep learning model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Evaluate the model on the test set
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

"""

Recommended action based on PPO: OFF


'class AIAgent:\n    def __init__(self, model_type: str, *args, **kwargs):\n        self.model_type = model_type\n\n        if model_type == \'ReinforcementLearning\':\n            # Initialize PPO model\n            self.policy = self.build_policy_network()\n            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)\n\n    def build_policy_network(self):\n        model = tf.keras.Sequential([\n            tf.keras.layers.Dense(64, activation=\'relu\', input_shape=(10,)),  # Assuming 11 features in the state\n            tf.keras.layers.Dense(2, activation=\'softmax\')  # Assuming binary action space (ON or OFF)\n        ])\n        return model\n  \n\n    def act(self, state: Dict[str, Any]) -> int:\n        # Convert state to numpy array\n        state_array = np.array(list(state.values()))\n        state_array = np.expand_dims(state_array, axis=0)\n\n        # Predict action probabilities\n        policy_prob = self.policy(state_array, training=False)\n\n        #  t

In [3]:
import tensorflow as tf
import numpy as np
from typing import List, Dict, Any

class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type

        if model_type == 'ReinforcementLearning':
            # Initialize PPO model
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def build_policy_network(self):
        model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(11,)),  # Assuming 11 features in the state
    tf.keras.layers.Dense(2, activation='softmax')
])

        return model

    def train_rl(self, dataset: List[Dict[str, Any]]):
    # Train the PPO model
        states = np.array([list(sample['input'].values()) for sample in dataset])
        actions = np.array([sample['label']['fan_state'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['label']['confidence']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions, num_classes=2)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def act(self, input_data: Dict[str, Any]) -> str:
        # Take an action based on the current state using the trained policy network
        state = np.array(list(input_data.values())).reshape(1, -1)
        policy_prob = self.policy(state, training=False).numpy()
        action = np.random.choice([0, 1], p=policy_prob.flatten())

        return 'ON' if action == 1 else 'OFF'

    def define_advantage(self, confidence: float) -> float:
        # Define advantage based on confidence
        return confidence


In [4]:
import tensorflow as tf
import numpy as np
from typing import List, Dict, Any

class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type

        if model_type == 'ReinforcementLearning':
            # Initialize PPO model
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(10,)),  # Assuming 11 features in the state
            tf.keras.layers.Dense(2, activation='softmax')  # Assuming binary action space (ON or OFF)
        ])
        return model
  

    def act(self, state: Dict[str, Any]) -> int:
        # Convert state to numpy array
        state_array = np.array(list(state.values()))
        state_array = np.expand_dims(state_array, axis=0)

        # Predict action probabilities
        policy_prob = self.policy(state_array, training=False)

        #  the action with the highest probability
        action = np.argmax(policy_prob[0])

        return 'ON' if action == 1 else 'OFF'

    def train_rl(self, dataset: List[Dict[str, Any]]):
    # Train the PPO model
        states = np.array([list(sample['input'].values()) for sample in dataset])
        print(states.shape)
        actions = np.array([sample['TargetFanState'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['confidence']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions, num_classes=2)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def define_advantage(self, confidence: float) -> float:
        return confidence

dummy_rl_dataset = [
    {
        'input': {
            'Temperature': 25.0,
            'TimeOfDay': 15.0,
            'Date': 5,
            'Month': 1,
            'DayOfWeek': 3,
            'MotionSensorState': 1,
            'ACTemperatureStatus': 0,
            'CurrentFanSpeed': 7,
            'Location': 2,
            'PreviousFanState': 1,
        },
        'TargetFanState': 1,  # Assuming the target fan state is ON
        'confidence': 0.9  
    }
]


ppo_agent = AIAgent(model_type='ReinforcementLearning')
ppo_agent.train_rl(dummy_rl_dataset)


current_state = {
    'Temperature': 25.0,
    'TimeOfDay': 15.0,
    'Date': 5,
    'Month': 1,
    'DayOfWeek': 3,
    'MotionSensorState': 1,
    'ACTemperatureStatus': 0,
    'CurrentFanSpeed': 7,
    'Location': 2,
    'PreviousFanState': 1,
}

action = ppo_agent.act(current_state)
print(f"Recommended action based on PPO: {action}")


(1, 10)
Recommended action based on PPO: ON


In [58]:
import tensorflow as tf
import numpy as np
from typing import List, Dict, Any, Tuple

class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type

        if model_type == 'DeepLearning':
            # Initialize deep learning model
            self.model = self.build_deep_learning_model()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
            self.model.compile(optimizer=self.optimizer, loss='binary_crossentropy', metrics=['accuracy'])

    def build_deep_learning_model(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(4,)),  # Assuming 4 features in the state
            tf.keras.layers.Dense(1, activation='sigmoid')  # Assuming binary output (ON or OFF)
        ])
        return model

    def train(self, dataset: List[Tuple[Dict[str, Any], int]]):
        # Train the deep learning model
        states = np.array([list(sample[0].values()) for sample in dataset])
        labels = np.array([sample[1] for sample in dataset])

        self.model.fit(states, labels, epochs=10, batch_size=32, verbose=1)

    def predict(self, input_data: Dict[str, Any]) -> Tuple[int, float]:
        # Predict the state and confidence based on the input data
        state = np.array(list(input_data.values())).reshape(1, -1)
        prediction = self.model.predict(state)
        confidence = prediction[0][0]  # Assuming binary output and getting the confidence for ON state

        return 1 if confidence > 0.85 else 0, confidence


deep_learning_agent = AIAgent(model_type='DeepLearning')

# Create a dummy dataset
dummy_deep_learning_dataset = [
    ({'temperature': 25.0, 'time': 15, 'motion': True, 'AC_temp_status': 22.0}, 1),
    ({'temperature': 20.0, 'time': 10, 'motion': False, 'AC_temp_status': 18.0}, 0),
    
]

# Train the Deep Learning model
deep_learning_agent.train(dummy_deep_learning_dataset)



current_state = {'temperature': 22.5, 'time': 12, 'motion': True, 'AC_temp_status': 20.0}
state, confidence = deep_learning_agent.predict(current_state)
print(f"Predicted state: {'ON' if state == 1 else 'OFF'}, Confidence: {confidence}")


Epoch 1/10
1/1 [==============================] - 1s 876ms/step - loss: 3.3477 - accuracy: 0.5000
Epoch 2/10
1/1 [==============================] - 0s 11ms/step - loss: 3.1827 - accuracy: 0.5000
Epoch 3/10
1/1 [==============================] - 0s 9ms/step - loss: 3.0181 - accuracy: 0.5000
Epoch 4/10
1/1 [==============================] - 0s 10ms/step - loss: 2.8551 - accuracy: 0.5000
Epoch 5/10
1/1 [==============================] - 0s 10ms/step - loss: 2.6984 - accuracy: 0.5000
Epoch 6/10
1/1 [==============================] - 0s 32ms/step - loss: 2.5423 - accuracy: 0.5000
Epoch 7/10
1/1 [==============================] - 0s 11ms/step - loss: 2.3867 - accuracy: 0.5000
Epoch 8/10
1/1 [==============================] - 0s 17ms/step - loss: 2.2320 - accuracy: 0.5000
Epoch 9/10
1/1 [==============================] - 0s 12ms/step - loss: 2.0784 - accuracy: 0.5000
Epoch 10/10
1/1 [==============================] - 0s 71ms/step
Predicted state: ON, Confidence: 0.9828900694847107


In [30]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# Function to create a dummy dataset
def create_dummy_dataset(num_samples=1000):
    np.random.seed(42)

    # Generate random data for the environment variables
    temperature = np.random.uniform(18.0, 30.0, size=num_samples)
    time_of_day = np.random.uniform(0.0, 24.0, size=num_samples)
    date = np.random.randint(1, 31, size=num_samples) # Assuming 31 days in a month 
    month = np.random.choice(12, size=num_samples)
    day_of_week = np.random.choice([2,3,4,5,6,7,1],#monday -2,tuesday-3,wednesday-4,thursday-5,friday-6,saturday-7,sunday-1
                                   size=num_samples)
    motion_sensor_state = np.random.choice([1,0], size=num_samples)#1-True,0-False
    ac_temperature_status = np.random.choice([1,0], size=num_samples) 

    # Generate random data for the fan variables
    current_fan_speed = np.random.randint(0, 11, size=num_samples)
    location = np.random.choice([1,2,3], size=num_samples) #1-Living Room,2-Bedroom,3-Kitchen
    previous_fan_state = np.random.choice([1,0,-1], size=num_samples)

    # Generate random data for the target variable
    target_fan_state = np.random.choice([1,0], size=num_samples)

    # Create a DataFrame
    df = pd.DataFrame({
        'Temperature': temperature,
        'TimeOfDay': time_of_day,
        'Date': date,
        'Month': month,
        'DayOfWeek': day_of_week,
        'MotionSensorState': motion_sensor_state,
        'ACTemperatureStatus': ac_temperature_status,
        'CurrentFanSpeed': current_fan_speed,
        'Location': location,
        'PreviousFanState': previous_fan_state,
        'TargetFanState': target_fan_state
    })

    return df

# Create a dummy dataset
dummy_dataset = create_dummy_dataset()

# Display the first few rows of the dataset
print(dummy_dataset.head())


# Create a dummy dataset
dummy_dataset = create_dummy_dataset()




   Temperature  TimeOfDay  Date  Month  DayOfWeek  MotionSensorState  \
0    22.494481   4.443190    28      7          6                  1   
1    29.408572  13.005623     7      9          3                  0   
2    26.783927  20.950700     3      0          1                  0   
3    25.183902  17.573397    23      7          4                  1   
4    19.872224  19.357468    28      6          6                  1   

   ACTemperatureStatus  CurrentFanSpeed  Location  PreviousFanState  \
0                    1                7         3                -1   
1                    0                5         3                -1   
2                    0                5         3                -1   
3                    1                9         3                -1   
4                    1                0         3                -1   

   TargetFanState  
0               0  
1               1  
2               1  
3               0  
4               1  


In [35]:
df = dummy_dataset

In [36]:
df.head()

,Temperature,TimeOfDay,Date,Month,DayOfWeek,MotionSensorState,ACTemperatureStatus,CurrentFanSpeed,Location,PreviousFanState,TargetFanState
0,22.494481,4.443190,28,7,6,1,1,7,3,-1,0
1,29.408572,13.005623,7,9,3,0,0,5,3,-1,1
2,26.783927,20.950700,3,0,1,0,0,5,3,-1,1
3,25.183902,17.573397,23,7,4,1,1,9,3,-1,0
4,19.872224,19.357468,28,6,6,1,1,0,3,-1,1


In [37]:

X = df.drop('TargetFanState', axis=1)
y = df['TargetFanState']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [38]:
# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Build a deep learning model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# Evaluate the model on the test set
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

Epoch 1/10
25/25 [==============================] - 1s 10ms/step - loss: 0.7170 - accuracy: 0.4787 - val_loss: 0.6988 - val_accuracy: 0.4700
Epoch 2/10
25/25 [==============================] - 0s 4ms/step - loss: 0.6932 - accuracy: 0.5300 - val_loss: 0.6975 - val_accuracy: 0.5100
Epoch 3/10
25/25 [==============================] - 0s 4ms/step - loss: 0.6857 - accuracy: 0.5450 - val_loss: 0.6979 - val_accuracy: 0.5050
Epoch 4/10
25/25 [==============================] - 0s 4ms/step - loss: 0.6792 - accuracy: 0.5650 - val_loss: 0.6987 - val_accuracy: 0.5200
Epoch 5/10
25/25 [==============================] - 0s 4ms/step - loss: 0.6732 - accuracy: 0.5725 - val_loss: 0.6997 - val_accuracy: 0.4950
Epoch 6/10
25/25 [==============================] - 0s 3ms/step - loss: 0.6696 - accuracy: 0.5950 - val_loss: 0.6997 - val_accuracy: 0.5350
Epoch 7/10
25/25 [==============================] - 0s 5ms/step - loss: 0.6643 - accuracy: 0.6062 - val_loss: 0.6986 - val_accuracy: 0.5450
Epoch 8/10
25/25 [=

In [25]:
import random

def create_dummy_dataset(num_samples: int) -> List[Dict[str, Union[Dict[str, Any], Dict[str, Union[str, float]]]]]:
    dataset = []

    for _ in range(num_samples):
        # Create a dummy input state
        input_state = {
            'temperature': random.uniform(20.0, 30.0),
            'time': random.randint(0, 23),
            'motion': random.choice([True, False]),
            'AC_temp_status': random.uniform(18.0, 25.0)
        }

        # Create a dummy label with confidence and fan state
        label = {
            'confidence': random.uniform(0.0, 1.0),
            'fan_state': random.choice([0, 1])  # 0 for OFF, 1 for ON
        }

        sample = {'input': input_state, 'label': label}
        dataset.append(sample)

    return dataset

# Example usage:
dummy_rl_dataset = create_dummy_dataset(100)


In [35]:
from typing import List, Dict, Union, Any, Tuple
from datetime import datetime, timedelta
import csv
import numpy as np
import tensorflow as tf

# Define the Fan class
class Fan:
    def __init__(self, fan_speed: int, location: str, *args, **kwargs):
        self.fan_speed = fan_speed 
        self.location = location

    def control_fan(self, speed: int):
        # Method to control fan speed
        self.fan_speed = speed

    def fetch_status(self) -> dict:
        # Method to fetch current status
        return {"Fan_Speed": self.fan_speed, "Location": self.location}

# Define the AIAgent class
class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type
        if model_type == 'ReinforcementLearning':
            # Initialize PPO model
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(4,)),  # Assuming 4 features in the state
            tf.keras.layers.Dense(2, activation='softmax')  # Assuming binary action space (ON or OFF)
        ])
        return model

    def train_rl(self, dataset: List[Dict[str, Any]]):
        # Train the PPO model
        states = np.array([list(sample['input'].values()) for sample in dataset])
        actions = np.array([sample['label']['fan_state'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['label']['confidence']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions, num_classes=2)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def act(self, input_data: Dict[str, Any]) -> str:
        # Take an action based on the current state using the trained policy network
        state = np.array(list(input_data.values())).reshape(1, -1)
        policy_prob = self.policy(state, training=False).numpy()
        action = np.random.choice([0, 1], p=policy_prob.flatten())

        return 'ON' if action == 1 else 'OFF'

    def predict(self, input_data: Dict[str, Any]) -> Tuple[int, float]:
        # Predict the state and confidence based on the input data
        # For simplicity, assume it returns a random state and confidence
        state = np.random.choice([0, 1])
        confidence = np.random.uniform(0, 1)
        return state, confidence

    def define_advantage(self, confidence: float) -> float:
        return confidence

# Define the Aggregator class
class Aggregator:
    def __init__(self, temperature, time, motion, AC_temp_status, *args, **kwargs):
        self.temperature = temperature
        self.time = time
        self.motion = motion
        self.AC_temp_status = AC_temp_status
        self.fan_objects = []
        self.ai_agent = None
        self.ai_output_log = []

    def define_fan_objects(self, fan_info: List[Dict[str, Union[int, str]]]):
        for info in fan_info:
            fan_obj = Fan(fan_speed=info["Fan_Speed"], location=info["Location"])
            self.fan_objects.append(fan_obj)

    def define_ai_object(self, ai_params: dict) -> AIAgent:
        self.ai_agent = AIAgent(model_type=ai_params.get('model_type', 'deep_learning'))
        return self.ai_agent

    def invoke_fan_control(self, fan_obj: Fan):
        if self.ai_agent:
            state, confidence = self.ai_agent.predict({"temperature": self.temperature, "motion": self.motion})
            if state == 1 and confidence >= 0.85:
                fan_obj.control_fan(5)  # Set fan speed to 5 if predicted state is ON with confidence >= 85%

    def save_ai_output_to_csv(self, ai_output: str):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.ai_output_log.append({"timestamp": timestamp, "output": ai_output})

        # Save the last 15 days' data
        last_15_days_data = [log for log in self.ai_output_log if datetime.now() - datetime.strptime(log["timestamp"], "%Y-%m-%d %H:%M:%S") <= timedelta(days=15)]

        with open('ai_output_log.csv', 'w', newline='') as csvfile:
            fieldnames = ["timestamp", "output"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(last_15_days_data)
    def save_fan_statuses(self, fan_statuses):
        with open(self.fan_status_file, 'a', newline='') as f:
            writer = csv.writer(f) 
            for status in fan_statuses:
                writer.writerow([status])
    def compute_difference(self, previous_state: dict, current_state: dict) -> bool:
        difference = any(previous_state.get(key) != current_state.get(key) for key in current_state)
        manual_override = difference
        return manual_override

    def compute_future_difference(self, current_state: dict, next_predicted_state: dict) -> bool:
        difference = any(current_state.get(key) != next_predicted_state.get(key) for key in current_state)
        return difference

    def send_signal(self):
        print("Sending signal")

    def fetch_status(self, fan_objects: List[Fan]):
        statuses = []
        for fan in fan_objects:
            statuses.append(fan.fetch_status())
        return statuses

    def run(self):
        # Define fan objects
        fan_info = [{"Fan_Speed": 0, "Location": "Living Room"}, {"Fan_Speed": 0, "Location": "Bedroom"}]
        self.define_fan_objects(fan_info)

        # Define AI agent
        ai_params = {"model_type": "ReinforcementLearning"}
        self.define_ai_object(ai_params)

        # Invoke fan control based on AI predictions
        for fan_obj in self.fan_objects:
            self.invoke_fan_control(fan_obj)
            self.save_ai_output_to_csv(fan_obj)
        # Save AI output to CSV
        self.save_ai_output_to_csv(fan_obj)

        # Fetch fan status
        fan_statuses = self.fetch_status(self.fan_objects)
        print("Fan statuses:", fan_statuses)

# Example usage:
aggregator = Aggregator(temperature=25.0, time=15, motion=True, AC_temp_status=22.0)
aggregator.run()



Fan statuses: [{'Fan_Speed': 0, 'Location': 'Living Room'}, {'Fan_Speed': 0, 'Location': 'Bedroom'}]


In [45]:
from typing import List, Dict, Union, Any, Tuple
from datetime import datetime, timedelta
import csv
import numpy as np
import tensorflow as tf

# Define the Fan class
class Fan:
    def __init__(self, fan_speed: int, location: str, *args, **kwargs):
        self.fan_speed = fan_speed 
        self.location = location

    def control_fan(self, speed: int):
        # Method to control fan speed
        self.fan_speed = speed

    def fetch_status(self) -> dict:
        # Method to fetch current status
        return {"Fan_Speed": self.fan_speed, "Location": self.location}

# Define the AIAgent class
class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type
        if model_type == 'ReinforcementLearning':
            # Initialize PPO model
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(4,)),  # Assuming 4 features in the state
            tf.keras.layers.Dense(2, activation='softmax')  # Assuming binary action space (ON or OFF)
        ])
        return model

    def train_rl(self, dataset: List[Dict[str, Any]]):
        # Train the PPO model
        states = np.array([list(sample['input'].values()) for sample in dataset])
        actions = np.array([sample['label']['fan_state'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['label']['confidence']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions, num_classes=2)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def act(self, input_data: Dict[str, Any]) -> str:
        # Take an action based on the current state using the trained policy network
        state = np.array(list(input_data.values())).reshape(1, -1)
        policy_prob = self.policy(state, training=False).numpy()
        action = np.random.choice([0, 1], p=policy_prob.flatten())

        return 'ON' if action == 1 else 'OFF'

    def predict(self, input_data: Dict[str, Any]) -> Tuple[int, float]:
        # Predict the state and confidence based on the input data
        # For simplicity, assume it returns a random state and confidence
        state = np.random.choice([0, 1])
        confidence = np.random.uniform(0, 1)
        return state, confidence

    def define_advantage(self, confidence: float) -> float:
        return confidence

# Define the Aggregator class
class Aggregator:
    def __init__(self, temperature, time, motion, AC_temp_status, *args, **kwargs):
        self.temperature = temperature
        self.time = time
        self.motion = motion
        self.AC_temp_status = AC_temp_status
        self.fan_objects = []
        self.ai_agent = None
        self.ai_output_log = []

    def define_fan_objects(self, fan_info: List[Dict[str, Union[int, str]]]):
        for info in fan_info:
            fan_obj = Fan(fan_speed=info["Fan_Speed"], location=info["Location"])
            self.fan_objects.append(fan_obj)

    def define_ai_object(self, ai_params: dict) -> AIAgent:
        self.ai_agent = AIAgent(model_type=ai_params.get('model_type', 'deep_learning'))
        return self.ai_agent

    def invoke_fan_control(self, fan_obj: Fan):
        if self.ai_agent:
            state, confidence = self.ai_agent.predict({"temperature": self.temperature, "motion": self.motion})
            if state == 1 and confidence >= 0.85:
                fan_obj.control_fan(5)  # Set fan speed to 5 if predicted state is ON with confidence >= 85%

    def save_ai_output_to_csv(self, ai_output: List[Dict[str, Union[int, float]]]):
      timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
      self.ai_output_log.append({"timestamp": timestamp, "output": ai_output})

      # Save the last 15 days' data
      last_15_days_data = [log for log in self.ai_output_log if datetime.now() - datetime.strptime(log["timestamp"], "%Y-%m-%d %H:%M:%S") <= timedelta(days=15)]

      with open('ai_output_log.csv', 'w', newline='') as csvfile:
          fieldnames = ["timestamp", "predicted_state", "confidence", "output"]  # Add 'output' field to fieldnames
          writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
          writer.writeheader()
          writer.writerows(last_15_days_data)


    def compute_difference(self, previous_state: dict, current_state: dict) -> bool:
        difference = any(previous_state.get(key) != current_state.get(key) for key in current_state)
        manual_override = difference
        return manual_override

    def compute_future_difference(self, current_state: dict, next_predicted_state: dict) -> bool:
        difference = any(current_state.get(key) != next_predicted_state.get(key) for key in current_state)
        return difference

    def send_signal(self):
        print("Sending signal")

    def fetch_status(self, fan_objects: List[Fan]):
        statuses = []
        for fan in fan_objects:
            statuses.append(fan.fetch_status())
        return statuses

    def run(self):
        # Define fan objects
        fan_info = [{"Fan_Speed": 0, "Location": "Living Room"}, {"Fan_Speed": 0, "Location": "Bedroom"},{"Fan_Speed": 0, "Location": "Kitchen"}]
        self.define_fan_objects(fan_info)

        # Define AI agent
        ai_params = {"model_type": "ReinforcementLearning"}
        self.define_ai_object(ai_params)

        # Invoke fan control based on AI predictions
        ai_output = []
        for fan_obj in self.fan_objects:
            predicted_state, confidence = self.ai_agent.predict({"temperature": self.temperature, "motion": self.motion})
            ai_output.append({"predicted_state": predicted_state, "confidence": confidence})
            if predicted_state == 1 and confidence >= 0.85:
                fan_obj.control_fan(5)  # Set fan speed to 5 if predicted state is ON with confidence >= 85%

        # Save AI output to CSV
        self.save_ai_output_to_csv(ai_output)

        # Fetch fan status
        fan_statuses = self.fetch_status(self.fan_objects)

        # Additional functionality can be added here as per requirements
        print(fan_statuses)
# Instantiate Aggregator and run
"""({'temperature': 25.0, 'time': 15, 'motion': True, 'AC_temp_status': 22.0}, 1),
    ({'temperature': 20.0, 'time': 10, 'motion': False, 'AC_temp_status': 18.0}, 0),"""
aggregator = Aggregator(temperature=20, time=10, motion=True, AC_temp_status= 18.0)
aggregator.run()


[{'Fan_Speed': 0, 'Location': 'Living Room'}, {'Fan_Speed': 0, 'Location': 'Bedroom'}, {'Fan_Speed': 0, 'Location': 'Kitchen'}]


In [49]:
aggregator = Aggregator(temperature=94, time=12, motion=True, AC_temp_status= 18.0)
aggregator.run()

[{'Fan_Speed': 0, 'Location': 'Living Room'}, {'Fan_Speed': 0, 'Location': 'Bedroom'}, {'Fan_Speed': 0, 'Location': 'Kitchen'}]


In [63]:
from typing import List, Dict, Union, Any, Tuple
import random
import datetime
import time
import json
import csv
from collections import deque
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
import secrets
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

#class Fan
class Fan:
    def __init__(self, fan_speed: int, location: str, *args, **kwargs):
        self.fan_speed = fan_speed 
        self.location = location
    def control_fan(self, speed: int):
        #  to control fan speed
        self.Fan_Speed = speed
    def fetch_status(self) -> dict:
        # to fetch current status
        return {"Fan_Speed": self.Fan_Speed, "Location": self.location}
    
    
class AIAgent:
    def __init__(self, model_type: str, *args, **kwargs):
        self.model_type = model_type

        if model_type == 'ReinforcementLearning':
            # Initialize PPO model
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(4,)),  # Assuming 4 features in the state
            tf.keras.layers.Dense(2, activation='softmax')  # Assuming binary action space (ON or OFF)
        ])
        return model

    def train_rl(self, dataset: List[Dict[str, Any]]):
        # Train the PPO model
        states = np.array([list(sample['input'].values()) for sample in dataset])
        actions = np.array([sample['label']['fan_state'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['label']['confidence']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions, num_classes=2)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def act(self, input_data: Dict[str, Any]) -> str:
        # Take an action based on the current state using the trained policy network
        state = np.array(list(input_data.values())).reshape(1, -1)
        policy_prob = self.policy(state, training=False).numpy()
        action = np.random.choice([0, 1], p=policy_prob.flatten())

        return 'ON' if action == 1 else 'OFF'

    def define_advantage(self, confidence: float) -> float:
       
        return confidence
    
    
#Aggregator class

class Aggregator:
    def __init__(self, temperature, time, motion, AC_temp_status, *args, **kwargs):
        self.temperature = temperature
        self.time = time
        self.motion = motion
        self.AC_temp_status = AC_temp_status
        self.fan_objects = []
        self.ai_agent = None
        self.ai_output_log = []

    def define_fan_objects(self, fan_info: List[Dict[str, Union[int, str]]]):
        for info in fan_info:
            fan_obj = Fan(Fan_Speed=info["Fan_Speed"], location=info["Location"])
            self.fan_objects.append(fan_obj)

    def define_ai_object(self, ai_params: dict) -> AIAgent:
        self.ai_agent = AIAgent(model_type=ai_params.get('model_type', 'deep_learning'))
        return self.ai_agent

    def invoke_fan_control(self, fan_obj: Fan):
        # fan control based on AI predictions
        if self.ai_agent:
            state, confidence = self.ai_agent.predict({"temperature": self.temperature, "motion": self.motion})
            if state == 1 and confidence >= 0.85:
                fan_obj.control_fan(5)  # Set fan speed to 5 if predicted state is ON with confidence >= 85%

    def save_ai_output_to_csv(self, ai_output: str):
        # Save output to CSV for the last 15 days
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.ai_output_log.append({"timestamp": timestamp, "output": ai_output})

        # Save the last 15 days' data
        last_15_days_data = [log for log in self.ai_output_log if datetime.now() - datetime.strptime(log["timestamp"], "%Y-%m-%d %H:%M:%S") <= timedelta(days=15)]

        with open('ai_output_log.csv', 'w', newline='') as csvfile:
            fieldnames = ["timestamp", "output"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(last_15_days_data)

    def compute_difference(self, previous_state: dict, current_state: dict) -> bool:
        #  difference between previous and current states
        difference = any(previous_state.get(key) != current_state.get(key) for key in current_state)
        manual_override = difference
        return manual_override

    def compute_future_difference(self, current_state: dict, next_predicted_state: dict) -> bool:
        #  difference between current and predicted states
        difference = any(current_state.get(key) != next_predicted_state.get(key) for key in current_state)
        return difference

    def send_signal(self):
        
        print("Sending signal")
    
    
    def fetch_status(self, fan_objects: List[Fan]):
        statuses = []
        for fan in fan_objects:
            statuses.append(fan.fetch_status())
        return statuses




    
ppo_agent = AIAgent(model_type='ReinforcementLearning')

# Train the PPO model
ppo_agent.train_rl(dummy_rl_dataset)

current_state = {'temperature': 25.0, 'time': 15, 'motion': True, 'AC_temp_status': 22.0}
action = ppo_agent.act(current_state)
print(f"Recommended action based on PPO: {action}")   

Recommended action based on PPO: ON


In [10]:
from typing import List, Dict, Union, Any,Tuple
import datetime
import csv
import numpy as np
import tensorflow as tf

class Fan:
    def __init__(self, fan_speed: int, location: str):
        self.fan_speed = fan_speed 
        self.location = location
        self.Fan_Speed = 0  
    
    def control_fan(self, speed: int):
        self.Fan_Speed = speed
    
    def fetch_status(self) -> dict:
        return {"Fan_Speed": self.Fan_Speed, "Location": self.location}

class Aggregator:
    def __init__(self):
        self.temperature = 0
        self.time = 0
        self.motion = False
        self.AC_temp_status = 0
        self.fan_objects = []
        self.ai_agent = None
        self.ai_output_log = []

    def add_fan(self, fan_speed: int, location: str):
        fan_obj = Fan(fan_speed, location)
        self.fan_objects.append(fan_obj)

    def define_ai_object(self, model_type: str) -> None:
        if model_type == 'ReinforcementLearning':
            self.ai_agent = AIAgent(model_type='ReinforcementLearning')

    def invoke_fan_control(self, fan_obj: Fan) -> None:
        # Invoke fan control based on AI predictions 
        if self.ai_agent:
            state, confidence = self.ai_agent.predict({"temperature": self.temperature, "motion": self.motion})
            if state == 1 and confidence >= 0.85:
                fan_obj.control_fan(5)  # Set fan speed to 5 if predicted state is ON with confidence >= 85% 

    def save_ai_output_to_csv(self) -> None:
        # Save AI output to CSV file
        with open('ai_output_log.csv', 'w', newline='') as csvfile:
            fieldnames = ["timestamp", "output"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(self.ai_output_log)

    def run(self, current_state: dict) -> List[dict]:
        # Run the aggregation process
        self.temperature = current_state.get('temperature', 0)
        self.time = current_state.get('time', 0)
        self.motion = current_state.get('motion', False)
        self.AC_temp_status = current_state.get('AC_temp_status', 0)

        # Invoke fan control for each fan object
        for fan_obj in self.fan_objects:
            self.invoke_fan_control(fan_obj)

        # Fetch status of all fan objects
        statuses = self.fetch_status(self.fan_objects)

        return statuses

    def fetch_status(self, fan_objects: List[Fan]) -> List[dict]:
        # Fetch status of fan objects
        statuses = []
        for fan in fan_objects:
            statuses.append(fan.fetch_status())
        return statuses

class AIAgent:
    def __init__(self, model_type: str):
        self.model_type = model_type

        if model_type == 'ReinforcementLearning':
            # PPO model
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(4,)),  # Assuming 4 features in the state
            tf.keras.layers.Dense(2, activation='softmax')  # Assuming binary action space (ON or OFF)
        ])
        return model

    def train_rl(self, dataset: List[Dict[str, Any]]) -> None:
        # Train the PPO model
        states = np.array([list(sample['input'].values()) for sample in dataset])
        actions = np.array([sample['label']['fan_state'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['label']['confidence']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions, num_classes=2)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def predict(self, input_data: Dict[str, Any]) -> Tuple[int, float]:
    # Add two dummy features to match the expected input shape
      input_data["feature_1"] = 0.0
      input_data["feature_2"] = 0.0

      # Convert input data to NumPy array
      state = np.array(list(input_data.values())).reshape(1, -1)

      # Predict using the policy network
      policy_prob = self.policy(state, training=False).numpy()
      action = np.random.choice([0, 1], p=policy_prob.flatten())
      confidence = np.max(policy_prob)

      return action, confidence

    def define_advantage(self, confidence: float) -> float:
        # Dummy implementation, replace with actual advantage computation
        return confidence



if __name__ == "__main__":
    # Example usage
    aggregator = Aggregator()
    aggregator.add_fan(fan_speed=3, location='Living Room')
    aggregator.add_fan(fan_speed=2, location='Bedroom')
    aggregator.define_ai_object(model_type='ReinforcementLearning')

    current_state = {'temperature': 25.0, 'time': 15, 'motion': True, 'AC_temp_status': 22.0}
    statuses = aggregator.run(current_state)
    print(statuses)

    # Example of saving AI output to CSV
    aggregator.ai_output_log.append({"timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "output": statuses})
    aggregator.save_ai_output_to_csv()

[{'Fan_Speed': 0, 'Location': 'Living Room'}, {'Fan_Speed': 0, 'Location': 'Bedroom'}]


In [3]:
import csv
import datetime
from typing import List, Dict, Any, Tuple
import numpy as np
import tensorflow as tf

class Fan:
    def __init__(self, fan_speed: int, location: str):
        self.fan_speed = fan_speed 
        self.location = location
        self.Fan_Speed = 0  # Initialize fan speed
    
    def control_fan(self, speed: int):
        # Control fan speed
        self.Fan_Speed = speed
    
    def fetch_status(self) -> dict:
        # Fetch current status
        return {"Fan_Speed": self.Fan_Speed, "Location": self.location}

class Aggregator:
    def __init__(self):
        self.temperature = 0
        self.time = 0
        self.motion = False
        self.AC_temp_status = 0
        self.fan_objects = []
        self.ai_agent = None
        self.ai_output_log = []

    def add_fan(self, fan_speed: int, location: str):
        # Add a new fan object
        fan_obj = Fan(fan_speed, location)
        self.fan_objects.append(fan_obj)

    def define_ai_object(self, model_type: str) -> None:
        if model_type == 'ReinforcementLearning':
            self.ai_agent = AIAgent(model_type='ReinforcementLearning')

    def invoke_fan_control(self, fan_obj: Fan) -> None:
        # Invoke fan control based on AI predictions
        if self.ai_agent:
            state, confidence = self.ai_agent.predict({"temperature": self.temperature, "motion": self.motion})
            if state == 1 and confidence >= 0.85:
                fan_obj.control_fan(5)  # Set fan speed to 5 if predicted state is ON with confidence >= 85%

    def save_ai_output_to_csv(self) -> None:
        # Save AI output to CSV file
        with open('ai_output_log.csv', 'w', newline='') as csvfile:
            fieldnames = ["timestamp", "output"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(self.ai_output_log)

    def run(self, current_state: dict) -> List[dict]:
        # Run the aggregation process
        self.temperature = current_state.get('temperature', 0)
        self.time = current_state.get('time', 0)
        self.motion = current_state.get('motion', False)
        self.AC_temp_status = current_state.get('AC_temp_status', 0)

        # Invoke fan control for each fan object
        for fan_obj in self.fan_objects:
            self.invoke_fan_control(fan_obj)

        # Fetch status of all fan objects
        statuses = self.fetch_status(self.fan_objects)
        
        # Save statuses to CSV file
        self.save_statuses_to_csv(statuses)

        return statuses

    def fetch_status(self, fan_objects: List[Fan]) -> List[dict]:
        # Fetch status of fan objects
        statuses = []
        for fan in fan_objects:
            statuses.append(fan.fetch_status())
        return statuses

    def save_statuses_to_csv(self, statuses: List[dict]) -> None:
        # Save statuses to CSV file
        with open('fan_statuses.csv', 'w', newline='') as csvfile:
            fieldnames = ["Location", "Fan_Speed"]
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for status in statuses:
                writer.writerow(status)

class AIAgent:
    def __init__(self, model_type: str):
        self.model_type = model_type

        if model_type == 'ReinforcementLearning':
            # Initialize PPO model
            self.policy = self.build_policy_network()
            self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def build_policy_network(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Dense(64, activation='relu', input_shape=(6,)),  # Assuming 6 features in the state (added 2 dummy features)
            tf.keras.layers.Dense(2, activation='softmax')  # Assuming binary action space (ON or OFF)
        ])
        return model

    def train_rl(self, dataset: List[Dict[str, Any]]) -> None:
        # Train the PPO model
        states = np.array([list(sample['input'].values()) for sample in dataset])
        actions = np.array([sample['label']['fan_state'] for sample in dataset])
        advantages = np.array([self.define_advantage(sample['label']['confidence']) for sample in dataset])

        actions_categorical = tf.keras.utils.to_categorical(actions, num_classes=2)

        with tf.GradientTape() as tape:
            policy_prob = self.policy(states, training=True)
            chosen_prob = tf.reduce_sum(actions_categorical * policy_prob, axis=1)
            old_policy_prob = tf.stop_gradient(chosen_prob)

            ratio = tf.exp(tf.math.log(chosen_prob + 1e-10) - tf.math.log(old_policy_prob + 1e-10))
            clipped_ratio = tf.clip_by_value(ratio, 1 - 0.2, 1 + 0.2)
            surrogate = -tf.minimum(ratio * advantages, clipped_ratio * advantages)
            loss = tf.reduce_mean(surrogate)

        gradients = tape.gradient(loss, self.policy.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.policy.trainable_variables))

    def predict(self, input_data: Dict[str, Any]) -> Tuple[int, float]:
        # Add two dummy features to match the expected input shape
        input_data["feature_1"] = 0.0
        input_data["feature_2"] = 0.0

        # Convert input data to NumPy array
        state = np.array(list(input_data.values())).reshape(1, -1)

        # Predict using the policy network
        policy_prob = self.policy(state, training=False).numpy()
        action = np.random.choice([0, 1], p=policy_prob.flatten())
        confidence = np.max(policy_prob)

        return action, confidence


if __name__ == "__main__":
    # Example usage
    aggregator = Aggregator()
    aggregator.add_fan(fan_speed=3, location='Living Room')
    aggregator.add_fan(fan_speed=2, location='Bedroom')
    aggregator.define_ai_object(model_type='ReinforcementLearning')

    current_state = {'temperature': 25.0, 'time': 15, 'motion': True, 'AC_temp_status': 22.0}
    statuses = aggregator.run(current_state)
    print(statuses)


ValueError: Input 0 of layer "sequential_2" is incompatible with the layer: expected shape=(None, 6), found shape=(1, 4)